# Credit Card Default Prediction — Modelling Pipeline

**Flow:**
1. Imports
2. Data Loading & Merging
3. Feature Definition & Train/Test Split
4. Imputation (fit on train only, apply to both)
5. Preprocessing Pipeline
6. Class Imbalance Analysis
7. Baseline Models (no resampling)
8. Imbalance-Handling Models (resampling + class-weight)
9. Unified Evaluation (confusion matrix + CV F1)
10. Threshold Sweep
11. Final Comparison

## Step 1 — Imports

In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier

import warnings
warnings.filterwarnings('ignore')

## Step 2 — Data Loading & Merging

In [14]:
csv1 = "../archive/Credit_Card_Dataset_2025_Sept_1.csv"
csv2 = "../archive/Credit_Card_Dataset_2025_Sept_2.csv"

df1 = pd.read_csv(csv1)
df2 = pd.read_csv(csv2)

df = pd.merge(df1, df2, left_on='ID', right_on='User', how='inner')

# Drop index-like / ID / low-value columns
df = df.drop(columns=[df.columns[0], "ID", "User", "FLAG_MOBIL"])

# Remove clearly invalid ages
df = df[df["AGE"] < 100]

print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['TARGET'].value_counts(normalize=True).round(4)}")
df.head()

Dataset shape: (25129, 17)
Target distribution:
TARGET
0    0.9832
1    0.0168
Name: proportion, dtype: float64


,GENDER,CAR,REALITY,NO_OF_CHILD,FAMILY_TYPE,HOUSE_TYPE,WORK_PHONE,PHONE,E_MAIL,FAMILY SIZE,BEGIN_MONTH,AGE,YEARS_EMPLOYED,TARGET,INCOME,INCOME_TYPE,EDUCATION_TYPE
0,M,Y,Y,0,Married,House / apartment,0,0,0,2.0,29,59,3.0,0,112500.0,Working,Secondary / secondary special
1,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,4,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special
2,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,26,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special
3,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,26,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special
4,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,38,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special


## Step 3 — Feature Definition & Train/Test Split

> **Split before any imputation** to prevent data leakage.

In [15]:
num_cols = ["NO_OF_CHILD", "BEGIN_MONTH", "AGE", "YEARS_EMPLOYED", "INCOME", "FAMILY SIZE"]
cat_cols = [
    "GENDER", "CAR", "REALITY", "FAMILY_TYPE", "HOUSE_TYPE",
    "WORK_PHONE", "PHONE", "E_MAIL", "INCOME_TYPE", "EDUCATION_TYPE"
]

X = df[num_cols + cat_cols]
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Train class balance:\n{y_train.value_counts(normalize=True).round(4)}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Train size: 20,103  |  Test size: 5,026
Train class balance:
TARGET
0    0.9832
1    0.0168
Name: proportion, dtype: float64


## Step 4 — Preprocessing Pipeline (with Imputation)

> **Fix:** Imputation is now **inside** the `ColumnTransformer` so it is fitted only on training folds during cross-validation — no leakage.  
> The original notebook computed medians on `X_train` then accidentally applied `X_train`'s mode to `X_test` (typo: `X_train[...].fillna(mode)` instead of `X_test[...]`). Both bugs are gone here.

In [16]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # handles YEARS_EMPLOYED, FAMILY SIZE
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # handles INCOME_TYPE
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

print("Preprocessor defined — imputation is leak-free inside the pipeline.")

Preprocessor defined — imputation is leak-free inside the pipeline.


## Step 5 — Class Imbalance Analysis

In [17]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Majority class (0): {neg:,}")
print(f"Minority class (1): {pos:,}")
print(f"Imbalance ratio   : {scale_pos_weight:.2f}x")
print(f"=> Use scale_pos_weight={scale_pos_weight:.2f} for XGBoost")

Majority class (0): 19,765
Minority class (1): 338
Imbalance ratio   : 58.48x
=> Use scale_pos_weight=58.48 for XGBoost


## Step 6 — Model Definitions

### 6a — Baseline Models (no resampling)

In [18]:
baseline_models = {
    "LogReg": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "LogReg_balanced": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42
        ))
    ]),
    "RandomForest": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        ))
    ]),
    "RandomForest_balanced": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
        ))
    ]),
    "GradientBoosting": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', GradientBoostingClassifier(
            n_estimators=200, random_state=42
        ))
    ]),
    "XGBoost": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=100, eval_metric='logloss', random_state=42
        ))
    ]),
    "XGBoost_weighted": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',
            random_state=42
        ))
    ]),
}

print(f"{len(baseline_models)} baseline models defined.")

7 baseline models defined.


### 6b — Imbalance-Handling Models (resampling + ensemble)

In [19]:
imb_models = {
    "SMOTE + LogReg": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('resampler', SMOTE(random_state=42)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "SMOTEENN + LogReg": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('resampler', SMOTEENN(random_state=42)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "RUS + LogReg": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('resampler', RandomUnderSampler(random_state=42)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Balanced RF": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', BalancedRandomForestClassifier(
            n_estimators=100, random_state=42
        ))
    ]),
    "Easy Ensemble": ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', EasyEnsembleClassifier(
            n_estimators=10, random_state=42
        ))
    ]),
}

print(f"{len(imb_models)} imbalance-handling models defined.")

5 imbalance-handling models defined.


## Step 7 — Evaluation Utilities

> **Fix:** `evaluate_models` now has a **single definition** used for all model groups (the original had two conflicting versions, one of which silently skipped CV).  
> Added ROC-AUC, F1 at threshold, and a threshold helper.

In [20]:
def predict_with_threshold(model, X, threshold=0.5):
    """Apply a custom probability threshold instead of the default 0.5."""
    y_probs = model.predict_proba(X)[:, 1]
    return (y_probs >= threshold).astype(int)


def evaluate_models(models_dict, label="", threshold=0.5, run_cv=True):
    """
    Fit each model on X_train, evaluate on X_test.
    Returns a styled DataFrame sorted by CV F1 (descending).
    """
    rows = []

    print(f"\n{'='*60}")
    print(f"  {label}  (threshold={threshold})")
    print(f"{'='*60}")

    for name, model in models_dict.items():
        print(f"\n▶ {name}")

        # --- Cross-validation on training set ---
        cv_f1 = np.nan
        cv_std = np.nan
        if run_cv:
            cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
            cv_f1, cv_std = cv_scores.mean(), cv_scores.std()
            print(f"   CV F1 : {cv_f1:.4f} ± {cv_std:.4f}")

        # --- Fit & predict on held-out test set ---
        model.fit(X_train, y_train)
        y_pred = predict_with_threshold(model, X_test, threshold)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)

        # ROC-AUC (uses raw probabilities, not threshold)
        y_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)

        print(f"   Test  — Precision: {precision:.4f}  Recall: {recall:.4f}  "
              f"F1: {f1:.4f}  AUC: {auc:.4f}")
        print(f"   TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}")

        rows.append({
            "Model": name,
            "CV F1 Mean": cv_f1,
            "CV F1 Std":  cv_std,
            "Test F1":    f1,
            "Precision":  precision,
            "Recall":     recall,
            "ROC-AUC":    auc,
            "TP":  tp,  "TN": tn,
            "FP":  fp,  "FN": fn,
        })

    results_df = pd.DataFrame(rows).sort_values("CV F1 Mean", ascending=False)
    return results_df


def style_results(df):
    return (df.style
            .background_gradient(cmap='RdYlGn',   subset=['TP', 'TN', 'Test F1','Precision', 'Recall', 'ROC-AUC'])
            .background_gradient(cmap='RdYlGn_r', subset=['FP', 'FN'])
            .format({
                "CV F1 Mean": "{:.4f}",
                "CV F1 Std":  "{:.4f}",
                "Test F1":    "{:.4f}",
                "Precision":  "{:.4f}",
                "Recall":     "{:.4f}",
                "ROC-AUC":    "{:.4f}",
                "TP":  "{:,}",  "TN": "{:,}",
                "FP":  "{:,}",  "FN": "{:,}",
            }))

print("Evaluation utilities ready.")

Evaluation utilities ready.


## Step 8 — Run Baseline Models

In [21]:
baseline_results = evaluate_models(baseline_models, label="BASELINE MODELS", threshold=0.5)
display(style_results(baseline_results))


  BASELINE MODELS  (threshold=0.5)

▶ LogReg
   CV F1 : 0.0288 ± 0.0257
   Test  — Precision: 1.0000  Recall: 0.0357  F1: 0.0690  AUC: 0.7178
   TP=3  TN=4,942  FP=0  FN=81

▶ LogReg_balanced
   CV F1 : 0.0521 ± 0.0005
   Test  — Precision: 0.0301  Recall: 0.6667  F1: 0.0576  AUC: 0.7116
   TP=56  TN=3,138  FP=1,804  FN=28

▶ RandomForest
   CV F1 : 0.2378 ± 0.0751
   Test  — Precision: 0.4333  Recall: 0.1548  F1: 0.2281  AUC: 0.7841
   TP=13  TN=4,925  FP=17  FN=71

▶ RandomForest_balanced
   CV F1 : 0.2052 ± 0.0674
   Test  — Precision: 0.4828  Recall: 0.1667  F1: 0.2478  AUC: 0.7688
   TP=14  TN=4,927  FP=15  FN=70

▶ GradientBoosting
   CV F1 : 0.1013 ± 0.0288
   Test  — Precision: 0.8333  Recall: 0.0595  F1: 0.1111  AUC: 0.7559
   TP=5  TN=4,941  FP=1  FN=79

▶ XGBoost
   CV F1 : 0.1585 ± 0.0456
   Test  — Precision: 0.5625  Recall: 0.1071  F1: 0.1800  AUC: 0.7510
   TP=9  TN=4,935  FP=7  FN=75

▶ XGBoost_weighted
   CV F1 : 0.1591 ± 0.0218
   Test  — Precision: 0.0799  Recall: 0

,Model,CV F1 Mean,CV F1 Std,Test F1,Precision,Recall,ROC-AUC,TP,TN,FP,FN
2,RandomForest,0.2378,0.0751,0.2281,0.4333,0.1548,0.7841,13,"4,925",17,71
3,RandomForest_balanced,0.2052,0.0674,0.2478,0.4828,0.1667,0.7688,14,"4,927",15,70
6,XGBoost_weighted,0.1591,0.0218,0.1364,0.0799,0.4643,0.7678,39,"4,493",449,45
5,XGBoost,0.1585,0.0456,0.1800,0.5625,0.1071,0.7510,9,"4,935",7,75
4,GradientBoosting,0.1013,0.0288,0.1111,0.8333,0.0595,0.7559,5,"4,941",1,79
1,LogReg_balanced,0.0521,0.0005,0.0576,0.0301,0.6667,0.7116,56,"3,138","1,804",28
0,LogReg,0.0288,0.0257,0.0690,1.0000,0.0357,0.7178,3,"4,942",0,81


## Step 9 — Run Imbalance-Handling Models

In [22]:
imb_results = evaluate_models(imb_models, label="IMBALANCE-HANDLING MODELS", threshold=0.5)
display(style_results(imb_results))


  IMBALANCE-HANDLING MODELS  (threshold=0.5)

▶ SMOTE + LogReg
   CV F1 : 0.0503 ± 0.0043
   Test  — Precision: 0.0282  Recall: 0.6190  F1: 0.0539  AUC: 0.6997
   TP=52  TN=3,149  FP=1,793  FN=32

▶ SMOTEENN + LogReg
   CV F1 : 0.0486 ± 0.0030
   Test  — Precision: 0.0296  Recall: 0.7024  F1: 0.0568  AUC: 0.7008
   TP=59  TN=3,008  FP=1,934  FN=25

▶ RUS + LogReg
   CV F1 : 0.0465 ± 0.0029
   Test  — Precision: 0.0293  Recall: 0.7143  F1: 0.0564  AUC: 0.7075
   TP=60  TN=2,957  FP=1,985  FN=24

▶ Balanced RF
   CV F1 : 0.1346 ± 0.0181
   Test  — Precision: 0.0772  Recall: 0.5595  F1: 0.1356  AUC: 0.8022
   TP=47  TN=4,380  FP=562  FN=37

▶ Easy Ensemble
   CV F1 : 0.0513 ± 0.0032
   Test  — Precision: 0.0302  Recall: 0.7143  F1: 0.0579  AUC: 0.7146
   TP=60  TN=3,015  FP=1,927  FN=24


,Model,CV F1 Mean,CV F1 Std,Test F1,Precision,Recall,ROC-AUC,TP,TN,FP,FN
3,Balanced RF,0.1346,0.0181,0.1356,0.0772,0.5595,0.8022,47,"4,380",562,37
4,Easy Ensemble,0.0513,0.0032,0.0579,0.0302,0.7143,0.7146,60,"3,015","1,927",24
0,SMOTE + LogReg,0.0503,0.0043,0.0539,0.0282,0.6190,0.6997,52,"3,149","1,793",32
1,SMOTEENN + LogReg,0.0486,0.0030,0.0568,0.0296,0.7024,0.7008,59,"3,008","1,934",25
2,RUS + LogReg,0.0465,0.0029,0.0564,0.0293,0.7143,0.7075,60,"2,957","1,985",24


## Step 10 — Threshold Sweep

> Sweep thresholds across **all** models in one unified pass.  
> CV is skipped here (models already fitted) so this is fast.

In [23]:
all_models = {**baseline_models, **imb_models}
thresholds  = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

sweep_rows = []
for t in thresholds:
    print(f"\n====== Threshold = {t} ======")
    for name, model in all_models.items():
        y_pred = predict_with_threshold(model, X_test, threshold=t)
        sweep_rows.append({
            "Threshold": t,
            "Model":     name,
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall":    recall_score(y_test, y_pred, zero_division=0),
            "F1":        f1_score(y_test, y_pred, zero_division=0),
        })

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df.sort_values(["Threshold", "F1"], ascending=[True, False]))


====== Threshold = 0.05 ======

====== Threshold = 0.1 ======

====== Threshold = 0.2 ======

====== Threshold = 0.3 ======

====== Threshold = 0.4 ======

====== Threshold = 0.5 ======


,Threshold,Model,Precision,Recall,F1
3,0.05,RandomForest_balanced,0.141379,0.488095,0.219251
5,0.05,XGBoost,0.143498,0.380952,0.208469
4,0.05,GradientBoosting,0.137755,0.321429,0.192857
2,0.05,RandomForest,0.106965,0.511905,0.176955
0,0.05,LogReg,0.097561,0.095238,0.096386
...,...,...,...,...,...
71,0.50,Easy Ensemble,0.030196,0.714286,0.057943
61,0.50,LogReg_balanced,0.030108,0.666667,0.057613
68,0.50,SMOTEENN + LogReg,0.029604,0.702381,0.056813
69,0.50,RUS + LogReg,0.029340,0.714286,0.056364


## Step 11 — Final Comparison (All Models, Default Threshold)

In [24]:
all_results = pd.concat([baseline_results, imb_results], ignore_index=True)
all_results = all_results.sort_values("CV F1 Mean", ascending=False).reset_index(drop=True)

print("\n=== FINAL LEADERBOARD ===")
display(style_results(all_results))


=== FINAL LEADERBOARD ===


,Model,CV F1 Mean,CV F1 Std,Test F1,Precision,Recall,ROC-AUC,TP,TN,FP,FN
0,RandomForest,0.2378,0.0751,0.2281,0.4333,0.1548,0.7841,13,"4,925",17,71
1,RandomForest_balanced,0.2052,0.0674,0.2478,0.4828,0.1667,0.7688,14,"4,927",15,70
2,XGBoost_weighted,0.1591,0.0218,0.1364,0.0799,0.4643,0.7678,39,"4,493",449,45
3,XGBoost,0.1585,0.0456,0.1800,0.5625,0.1071,0.7510,9,"4,935",7,75
4,Balanced RF,0.1346,0.0181,0.1356,0.0772,0.5595,0.8022,47,"4,380",562,37
5,GradientBoosting,0.1013,0.0288,0.1111,0.8333,0.0595,0.7559,5,"4,941",1,79
6,LogReg_balanced,0.0521,0.0005,0.0576,0.0301,0.6667,0.7116,56,"3,138","1,804",28
7,Easy Ensemble,0.0513,0.0032,0.0579,0.0302,0.7143,0.7146,60,"3,015","1,927",24
8,SMOTE + LogReg,0.0503,0.0043,0.0539,0.0282,0.6190,0.6997,52,"3,149","1,793",32
9,SMOTEENN + LogReg,0.0486,0.0030,0.0568,0.0296,0.7024,0.7008,59,"3,008","1,934",25
